# `ciclo`: control del ciclo semanal de extracción

Notebook de utilidades. Define el estado del ciclo semanal en la tabla Delta `workspace.inferiores.bronze_ciclos` y las funciones que usan los demás notebooks del pipeline para decidir si les toca correr. No procesa datos de jugadores ni de clubes.

## Tabla `bronze_ciclos`

Una fila por ciclo. Todas las columnas son `STRING`.

| Columna | Contenido |
|---|---|
| `lunes` | Fecha ISO del lunes de la semana del ciclo, en hora de Argentina |
| `inicio` | Momento en que se abrió el ciclo |
| `extraccion_en` | Momento en que terminó la extracción de clubes y movimientos |
| `cerrado_en` | Momento en que terminó la bajada de todos los jugadores |
| `publicado_en` | Momento en que se publicaron silver y gold |

Orden de los hitos dentro de un ciclo: `inicio` → `extraccion_en` → `cerrado_en` → `publicado_en`.

## Contenido de la celda

| Elemento | Qué hace |
|---|---|
| `FORZAR` (widget de texto, por defecto `0`) | Con valor `1`, `motivo_no_corre` devuelve siempre `None` y el notebook que lo consulta corre ignorando el estado del ciclo. Pensado para corridas manuales |
| `CREATE TABLE IF NOT EXISTS` | Crea `bronze_ciclos` si no existe |
| `ahora()` | Hora actual en formato ISO, con precisión de segundos. El comentario del código indica que es la misma hora (UTC) que usa `bronze_player_raw` |
| `forzar()` | `True` si el widget `FORZAR` vale `1`; `False` si vale otra cosa o si el widget no existe |
| `lunes_ar()` | Fecha ISO del lunes de la semana actual según la zona `America/Argentina/Buenos_Aires` |
| `ciclo_vigente()` | Devuelve como diccionario la fila con `inicio` más reciente, o `None` si la tabla está vacía |
| `abrir_ciclo_si_corresponde()` | Inserta un ciclo nuevo si no hay ninguno, o si el último ya se publicó y empezó una semana nueva. Si el último sigue abierto, lo devuelve sin crear otro, aunque haya pasado el domingo |
| `marcar(campo)` | Escribe la hora actual en `extraccion_en`, `cerrado_en` o `publicado_en` del ciclo vigente, solo si el campo estaba vacío. `publicado_en` además exige que `cerrado_en` ya tenga valor |
| `motivo_no_corre(paso)` | Con `paso` = `'jugadores'` o `'publicar'`, devuelve el texto de por qué no hay que correr, o `None` si hay que correr |

## Reglas de `motivo_no_corre`

| Paso | No corre si |
|---|---|
| (cualquiera) | No hay ciclo abierto. Lo abre `01_extraccion` |
| `jugadores` | `extraccion_en` está vacío, o `cerrado_en` ya tiene valor |
| `publicar` | `cerrado_en` está vacío, o `publicado_en` ya tiene valor |

Con `FORZAR` = `1` ninguna de estas reglas se aplica.

## Quién lo usa

| Notebook | Usa |
|---|---|
| `01_extraccion` | `abrir_ciclo_si_corresponde()`, `forzar()` y `marcar("extraccion_en")` |
| `01b_primer_equipo` | `motivo_no_corre("jugadores")`, `ciclo_vigente()` y `ahora()` |
| `02_jugadores` | `motivo_no_corre("jugadores")`, `ciclo_vigente()`, `marcar("cerrado_en")` y `ahora()` |
| `03_silver` | `motivo_no_corre("publicar")` |
| `clasificar_clubes_externos` | `motivo_no_corre("publicar")` |
| `clasificar_competiciones` | `motivo_no_corre("publicar")` |
| `04_gold` | `motivo_no_corre("publicar")` y `marcar("publicado_en")` |
| `05_calidad` | `ciclo_vigente()` y `forzar()` |


In [0]:
from datetime import datetime as _dt, timedelta as _td
from zoneinfo import ZoneInfo
from pyspark.sql.types import StructType, StructField, StringType

CICLOS = "workspace.inferiores.bronze_ciclos"
TZ_AR = ZoneInfo("America/Argentina/Buenos_Aires")
SCHEMA_CICLOS = StructType([StructField(c, StringType()) for c in
                            ["lunes", "inicio", "extraccion_en", "cerrado_en", "publicado_en"]])

dbutils.widgets.text("FORZAR", "0")   # 1 = correr a mano ignorando el estado del ciclo

spark.sql(f"""CREATE TABLE IF NOT EXISTS {CICLOS}
              (lunes STRING, inicio STRING, extraccion_en STRING, cerrado_en STRING, publicado_en STRING)
              USING DELTA""")

def ahora():
    return _dt.now().isoformat(timespec="seconds")   # misma hora (UTC) que usa bronze_player_raw

def forzar():
    try:
        return dbutils.widgets.get("FORZAR") == "1"
    except Exception:
        return False

def lunes_ar():
    hoy = _dt.now(TZ_AR).date()
    return (hoy - _td(days=hoy.weekday())).isoformat()

def ciclo_vigente():
    f = spark.sql(f"SELECT * FROM {CICLOS} ORDER BY inicio DESC LIMIT 1").collect()
    return f[0].asDict() if f else None

def abrir_ciclo_si_corresponde():
    # Ciclo nuevo solo si el ultimo ya se publico y empezo una semana nueva (lunes mas nuevo).
    # Si el ultimo sigue abierto, se sigue con ese aunque pase el domingo.
    c = ciclo_vigente()
    if c is None or (c["publicado_en"] and lunes_ar() > c["lunes"]):
        spark.createDataFrame([(lunes_ar(), ahora(), None, None, None)], SCHEMA_CICLOS) \
             .write.mode("append").saveAsTable(CICLOS)
        c = ciclo_vigente()
        print(f"CICLO NUEVO: semana del {c['lunes']}")
    return c

def marcar(campo):
    # campo: extraccion_en | cerrado_en | publicado_en (solo escribe si estaba vacio)
    c = ciclo_vigente()
    extra = " AND cerrado_en IS NOT NULL" if campo == "publicado_en" else ""
    spark.sql(f"UPDATE {CICLOS} SET {campo} = '{ahora()}' "
              f"WHERE inicio = '{c['inicio']}' AND {campo} IS NULL{extra}")

def motivo_no_corre(paso):
    # paso: 'jugadores' o 'publicar'. Devuelve por que NO hay que correr, o None si hay que correr.
    if forzar():
        return None
    c = ciclo_vigente()
    if c is None:
        return "no hay ciclo abierto (lo abre 01_extraccion)"
    if paso == "jugadores":
        if not c["extraccion_en"]:
            return f"ciclo {c['lunes']}: falta terminar la extraccion"
        if c["cerrado_en"]:
            return f"ciclo {c['lunes']}: jugadores ya completos"
    if paso == "publicar":
        if not c["cerrado_en"]:
            return f"ciclo {c['lunes']}: todavia faltan jugadores"
        if c["publicado_en"]:
            return f"ciclo {c['lunes']}: ya publicado"
    return None